# Music Annotation Project V8.8E
## Notebook 56: Supervised M5_REMOVE Pilot Reviewer

**Runtime:** CPU. Select **Runtime → Run all**.

This notebook presents V8.8 correction candidates to a human reviewer after Notebook 55 has passed. It does not auto-approve outputs. The available page decisions are:

- **Accept V8.8 correction**
- **Retain Notebook 09 output**
- **Defer**
- **Reject page / require upstream mask review**

Final approval is blocked while any page is unreviewed or deferred. The notebook records immutable input identities, every decision transition, approved output hashes and the pilot burden/outcome summary.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy gradio pillow


In [ ]:
import gradio as gr
from PIL import Image
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N53_ROOT=ROOT/'m5_remove_operational_builder_v8_8b'
N55_ROOT=ROOT/'m5_remove_notebook09_shadow_v8_8d'
OUTPUT_ROOT=ROOT/'m5_remove_supervised_pilot_v8_8e'
NOTEBOOK53_RUN_OVERRIDE=None
NOTEBOOK55_RUN_OVERRIDE=None
PUBLIC_LINK=True
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook56_v8_8e_pilot'
OUT=OUTPUT_ROOT/RUN; DEC=OUT/'decisions'; FINAL=OUT/'final'; T=OUT/'tables'; A=OUT/'audit'; U=OUT/'upload'
for p in [OUT,DEC,FINAL,T,A,U]:p.mkdir(parents=True,exist_ok=False)


## 1. Resolve passed Notebook 55 and its exact Notebook 53 source

In [ ]:
N55=Path(NOTEBOOK55_RUN_OVERRIDE) if NOTEBOOK55_RUN_OVERRIDE else latest_complete(N55_ROOT,['notebook55_verdict.json','tables/shadow_page_comparison.csv'])
require(N55 is not None,'Complete Notebook 55 run not found.')
v55=read_json(N55/'notebook55_verdict.json'); require(v55.get('verdict')=='PASS_SHADOW_EVIDENCE_READY_FOR_SUPERVISED_PILOT',f'Notebook 55 did not pass: {v55}')
N53=Path(NOTEBOOK53_RUN_OVERRIDE) if NOTEBOOK53_RUN_OVERRIDE else Path(v55['notebook53_run'])
require(N53.exists(),'Notebook 53 source recorded by Notebook 55 is unavailable.')
N09=Path(v55['notebook09_run']); require(N09.exists(),'Notebook 09 source recorded by Notebook 55 is unavailable.')
compare=pd.read_csv(N55/'tables'/'shadow_page_comparison.csv'); pages=sorted(compare.page.astype(str).tolist()); require(len(pages)>0,'Pilot roster empty.')
policy_id=v55['policy_id']; implementation_id=v55['implementation_id']
print('Pages:',len(pages));print('Policy:',policy_id)


## 2. Load page pairs and initialise durable review state

In [ ]:
def incumbent_path(page):
    hits=list((N09/'final').glob(page+'.*')); require(len(hits)==1,f'{page}: expected one Notebook09 final page, found {hits}'); return hits[0]
def v88_path(page):
    p=N53/'pages'/page/'corrected.png'; require(p.exists(),f'{page}: V8.8 corrected page missing'); return p
STATE={}
for page in pages:
    a=imread(incumbent_path(page)); b=imread(v88_path(page)); obs=imread(N53/'pages'/page/'observed.png'); m3=imread(N53/'pages'/page/'m3.png')>0; sr=imread(N53/'pages'/page/'safe_remove.png')>0
    require(a.shape==b.shape==obs.shape==m3.shape==sr.shape,f'{page}: shape mismatch')
    STATE[page]={'status':'unreviewed','notes':'','history':[],'notebook09':a,'v88':b,'observed':obs,'m3':m3,'safe_remove':sr,'notebook09_path':str(incumbent_path(page)),'v88_path':str(v88_path(page))}

def persist(page,event):
    s=STATE[page]; atomic_json(DEC/f'{page}_decision.json',{'page':page,'status':s['status'],'notes':s['notes'],'history':s['history'],'event':event,'policy_id':policy_id,'implementation_id':implementation_id,'notebook09_sha256':array_hash(s['notebook09']),'v88_sha256':array_hash(s['v88']),'updated_utc':datetime.now(timezone.utc).isoformat()})
for page in pages:persist(page,{'event':'initialise'})


## 3. Reviewer functions

In [ ]:
def counts():
    vals=[s['status'] for s in STATE.values()]; return {k:vals.count(k) for k in ['unreviewed','accept_v88','retain_notebook09','defer','mask_review_required']}
def pair_view(page,overlay=True):
    s=STATE[page]; a=cv2.cvtColor(s['notebook09'],cv2.COLOR_GRAY2RGB); b=cv2.cvtColor(s['v88'],cv2.COLOR_GRAY2RGB)
    if overlay:
        d=s['notebook09']!=s['v88']; b[d]=(255,40,40); b[s['safe_remove']]=(40,190,70)
    divider=np.full((a.shape[0],8,3),235,np.uint8); out=np.hstack([a,divider,b]);
    cv2.putText(out,'NOTEBOOK 09',(12,28),cv2.FONT_HERSHEY_SIMPLEX,.7,(30,30,30),2,cv2.LINE_AA); cv2.putText(out,'V8.8',(a.shape[1]+20,28),cv2.FONT_HERSHEY_SIMPLEX,.7,(30,30,30),2,cv2.LINE_AA)
    return out
def info(page):
    s=STATE[page]; d=s['notebook09']!=s['v88']; c=counts(); return f"{page} | status={s['status']} | disagreement={int(d.sum())} pixels | SAFE_REMOVE={int(s['safe_remove'].sum())} | queue: {c}"
def refresh(page,overlay=True): return pair_view(page,overlay),info(page),STATE[page]['notes']
def decide(page,status,notes,overlay=True):
    require(status in ['accept_v88','retain_notebook09','defer','mask_review_required'],'Invalid decision')
    s=STATE[page]; s['status']=status;s['notes']=notes;s['history'].append({'utc':datetime.now(timezone.utc).isoformat(),'status':status,'notes':notes});persist(page,{'event':'decision','status':status});return refresh(page,overlay)
def approve_all(notes):
    c=counts(); require(c['unreviewed']==0 and c['defer']==0,f'Pilot cannot close: {c}')
    rows=[]
    for page,s in STATE.items():
        if s['status']=='accept_v88': out=s['v88']; source='V8.8'
        elif s['status'] in ['retain_notebook09','mask_review_required']: out=s['notebook09']; source='Notebook09'
        else: raise RuntimeError(f'{page}: unresolved status {s["status"]}')
        path=FINAL/f'{page}.png'; require(cv2.imwrite(str(path),out),f'{page}: final write failed')
        rows.append({'page':page,'decision':s['status'],'approved_source':source,'notes':s['notes'],'final_path':str(path),'final_sha256':sha256_file(path),'v88_disagreement_pixels':int(np.sum(s['notebook09']!=s['v88'])),'safe_remove_pixels':int(s['safe_remove'].sum())})
    result=pd.DataFrame(rows);result.to_csv(T/'pilot_page_decisions.csv',index=False)
    atomic_json(OUT/'pilot_approval.json',{'approved_utc':datetime.now(timezone.utc).isoformat(),'approval_notes':notes,'policy_id':policy_id,'implementation_id':implementation_id,'counts':counts(),'pages':len(result),'v88_accepted':int((result.approved_source=='V8.8').sum()),'notebook09_retained':int((result.approved_source=='Notebook09').sum()),'automatic_approval':False})
    return f'Pilot closed. {counts()}'


## 4. Supervised reviewer

In [ ]:
first=pages[0]
with gr.Blocks(title='V8.8E Supervised M5_REMOVE Pilot') as app:
    gr.Markdown('# V8.8E supervised M5_REMOVE pilot\nLeft: Notebook 09. Right: V8.8. Red indicates disagreement; green indicates SAFE_REMOVE. Human approval remains mandatory.')
    with gr.Row(): page=gr.Dropdown(pages,value=first,label='Page'); overlay=gr.Checkbox(True,label='Show disagreement / SAFE_REMOVE overlay')
    view=gr.Image(height=760,label='Notebook 09 | V8.8'); status=gr.Textbox(interactive=False,label='Page and queue status'); notes=gr.Textbox(label='Decision notes')
    with gr.Row(): accept=gr.Button('Accept V8.8 correction',variant='primary'); retain=gr.Button('Retain Notebook 09 output'); defer=gr.Button('Defer'); maskreview=gr.Button('Require mask review')
    gr.Markdown('## Close pilot'); approval_notes=gr.Textbox(label='Pilot approval notes'); close=gr.Button('Write approved pages and close pilot',variant='primary'); close_status=gr.Textbox(interactive=False)
    page.change(refresh,[page,overlay],[view,status,notes]); overlay.change(refresh,[page,overlay],[view,status,notes]); app.load(refresh,[page,overlay],[view,status,notes])
    accept.click(lambda p,n,o:decide(p,'accept_v88',n,o),[page,notes,overlay],[view,status,notes]); retain.click(lambda p,n,o:decide(p,'retain_notebook09',n,o),[page,notes,overlay],[view,status,notes]); defer.click(lambda p,n,o:decide(p,'defer',n,o),[page,notes,overlay],[view,status,notes]); maskreview.click(lambda p,n,o:decide(p,'mask_review_required',n,o),[page,notes,overlay],[view,status,notes]); close.click(approve_all,[approval_notes],[close_status])
app.launch(share=PUBLIC_LINK,debug=False,show_error=True)


## 5. Run after the reviewer closes the pilot

In [ ]:
require((OUT/'pilot_approval.json').exists(),'Pilot is not closed. Complete decisions and use the close button first.')
decisions=pd.read_csv(T/'pilot_page_decisions.csv'); approval=read_json(OUT/'pilot_approval.json')
gates={'notebook55_pass':True,'all_pages_decided':len(decisions)==len(pages),'no_unreviewed':counts()['unreviewed']==0,'no_deferred':counts()['defer']==0,'final_pages_complete':all((FINAL/f'{p}.png').exists() for p in pages),'human_approval_recorded':True}
first_failed=next((k for k,v in gates.items() if v is not True),None)
verdict='PASS_SUPERVISED_PILOT_COMPLETE' if first_failed is None else 'FAIL_V8_8E_PILOT'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':verdict,'first_failed_gate':first_failed,'gates':gates,'policy_id':policy_id,'implementation_id':implementation_id,'notebook55_run':str(N55),'notebook53_run':str(N53),'pages':len(pages),'v88_accepted':int((decisions.approved_source=='V8.8').sum()),'notebook09_retained':int((decisions.approved_source=='Notebook09').sum()),'mask_review_required':int((decisions.decision=='mask_review_required').sum()),'automatic_approval':False,'promotion_authority':False,'meaning':'Completion records supervised pilot outcomes. A separate release decision is required before changing the primary correction engine.'}
atomic_json(OUT/'notebook56_verdict.json',record); print(json.dumps(record,indent=2))
if first_failed:raise RuntimeError(f'Notebook 56 failed at {first_failed}')


## 6. Curated upload package

In [ ]:
summary=['NOTEBOOK 56 / V8.8E SUPERVISED M5_REMOVE PILOT','',f'Verdict: {verdict}',f'Pages: {len(pages)}',f'V8.8 accepted: {record["v88_accepted"]}',f'Notebook 09 retained: {record["notebook09_retained"]}',f'Mask review required: {record["mask_review_required"]}',f'Policy ID: {policy_id}','','Human final approval was mandatory. This notebook has no production-promotion authority.']
(OUT/'executive_summary.txt').write_text(chr(10).join(summary))
files=[OUT/'executive_summary.txt',OUT/'notebook56_verdict.json',OUT/'pilot_approval.json',T/'pilot_page_decisions.csv']+sorted(DEC.glob('*.json'))
rows=[]
for src in files:
    require(src.exists(),f'Missing upload artefact {src}'); dst=U/src.name; shutil.copy2(src,dst); rows.append({'file':dst.name,'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(rows).to_csv(U/'upload_manifest.csv',index=False); (U/'README.txt').write_text('Notebook 56 V8.8E supervised pilot decision pack.'+chr(10))
zp=U/'notebook56_v8_8e_upload_bundle.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for p in U.iterdir():
        if p.is_file() and p!=zp:z.write(p,p.name)
require(zp.stat().st_size>0,'Upload ZIP failed'); print('Upload:',zp)
